# Loanword glossary builder

Find the Perso-Arabic vocabulary in a Hindi passage and offer it as a ranked list of
candidates for an editor to accept or reject.

**This notebook has not been run against the live API.** Every code cell below ships
with an empty output. There was no API key available when it was written, so nothing
was executed and no result was invented. Run the cells yourself to see real output.

Pipeline overview:

1. Normalise the passage to NFD and split it into words.
2. Find every nukta and report which letter it sits under.
3. Score every distinct word on rarity, word ending and nukta.
4. Rank what clears the threshold and render a plain-text appendix.
5. Ask `sarvam-105b` for one short meaning per candidate (the only step needing a key).

Steps 1 to 4 are pure Python with no network call and no key.

**A nukta is not a loanword marker.** It marks only the q, kh, gh, z and f sounds, so
`किताब` (kitab) is an Arabic borrowing carrying no nukta at all, and the scorer is what
catches it. In the other direction `ड़` and `ढ़` carry a nukta in ordinary native words
like `बड़ा` and `घोड़ा`, which are not borrowings. What comes out of this notebook is a
candidate list for an editor, never a verdict.

In [ ]:
%pip install -r requirements.txt

## Setup

The analysis layer imports only the Python standard library. The key is read here
because the last step needs it, and it is passed to the client explicitly rather than
left to the client's default, which is frozen at import time and would be `None`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd()))

import loanword_glossary as lg

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your .env file before running the gloss step. "
        "Steps 1 to 4 below run without it."
    )

## 1. The passage

`SAMPLE_PASSAGE` is original Hindi prose written for this recipe. It is not a quotation
and is attributed to nobody. Replace it with your own text whenever you like: read a
file with `Path(...).read_text(encoding="utf-8")` and pass that in instead.

It lives in the module rather than in `sample_data/`, because `sample_data/` is
gitignored in every recipe here and nothing placed there would ship.

In [ ]:
text = lg.SAMPLE_PASSAGE

tokens = lg.tokenize(text)
counts = lg.word_counts(text)

print(text)
print()
print("characters:", len(text))
print("words:", len(tokens))
print("distinct words:", len(counts))

## 2. The nukta detector, and both of its blind spots

`nukta_marks` reports every nukta and the letter under it. `has_perso_arabic_nukta` is
the decision, and it is true only for the five borrowed letters.

Read the two blocks below together. The first word is a borrowing the detector cannot
see. The second is an everyday native word the naive "any dot" rule would flag.

In [ ]:
for word in ("फ़ौज", "ज़मीन", "क़लम", "ग़ज़ल", "ख़बर"):
    marks = lg.nukta_marks(word)
    print(word, "->", [(m.base, m.origin) for m in marks],
          "perso-arabic:", lg.has_perso_arabic_nukta(word))

print()
print("the recall gap - a borrowing with no mark at all:")
for word in ("किताब", "जवाब", "हिसाब", "दुकानदार"):
    print(" ", word, "marks:", lg.nukta_marks(word),
          "perso-arabic:", lg.has_perso_arabic_nukta(word))

print()
print("the precision gap - native words that DO carry a mark:")
for word in ("बड़ा", "पढ़ना", "लड़का", "घोड़ा", "कपड़े"):
    marks = lg.nukta_marks(word)
    print(" ", word, "marks:", [(m.base, m.origin) for m in marks],
          "perso-arabic:", lg.has_perso_arabic_nukta(word))

## 3. The scorer

Three terms, added and capped at 1.0, with a common-word veto in front:

    rarity  0.40 / count      word ending  0.40      nukta  0.55

Anything at or above `0.55` becomes a candidate. The numbers below are the boundary the
whole design turns on: a word with a Perso-Arabic ending qualifies at one or two
occurrences and drops out at three.

In [ ]:
from collections import Counter

kitab = lg.normalise("किताब")
for times in (1, 2, 3, 4, 200):
    value = lg.score(kitab, Counter({kitab: times}))
    print("kitab seen %3d time(s): %.3f  %s"
          % (times, value, "candidate" if value >= lg.CANDIDATE_THRESHOLD else "-"))

print()
fauj = lg.normalise("फ़ौज")
for times in (1, 20, 200):
    value = lg.score(fauj, Counter({fauj: times}))
    print("fauj  seen %3d time(s): %.3f  %s"
          % (times, value, "candidate" if value >= lg.CANDIDATE_THRESHOLD else "-"))

print()
print("rejected endings, kept in the code with the native words that killed them:")
for ending, casualties in lg.REJECTED_SUFFIXES.items():
    print(" ", ending, "would flag:", casualties)

## 4. Rank, and render the appendix

Sorted by score, then by first appearance in the passage, then by the word itself, so
the output is the same on every run.

The `gloss` line reads `(not generated - no API key)` until the next step fills it in.
That is deliberate: a half-finished appendix should not look like a complete one.

In [ ]:
candidates = lg.rank_candidates(text)

for candidate in candidates:
    print("%.3f  %-12s count=%d  marks=%d  ending=%s"
          % (candidate.score, candidate.surface, candidate.count,
             len(candidate.marks), candidate.suffix))

print()
print("candidates with no nukta anywhere - what the scorer adds over the detector:")
print(" ", [c.surface for c in candidates if not c.marks])

In [ ]:
print(lg.render_appendix(candidates))

## 5. One short meaning per word

The only step that makes a network call. `sarvam-105b` is asked for a one-line meaning
per word and is told to answer `unknown` rather than guess. It is never asked where a
word came from: this tool has no way to check an etymology, and a plausible wrong one is
worse than none.

`gloss_candidates` raises rather than returning a partial answer if the model refuses or
if the number of lines back does not match the number of words sent.

In [ ]:
import sarvam_glossing as sg

client = sg.make_client()

print(sg.build_gloss_prompt(candidates))

In [ ]:
glosses = sg.gloss_candidates(client, candidates)

appendix = lg.render_appendix(candidates, glosses=glosses)
print(appendix)

## 6. Save it

Written to `outputs/`, which is gitignored, so nothing you generate here is committed by
accident.

In [ ]:
output_path = Path("outputs") / "appendix.txt"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(appendix, encoding="utf-8")

print("wrote", output_path, "-", len(appendix), "characters")

## Where to take it next

Point it at a real book. Premchand is the natural target: he died in 1936, Indian
copyright runs for the author's life plus 60 years, so his work entered the public
domain on 1 January 1997. None of his words ship here, because there is no way to verify
offline that a passage typed from memory matches what he wrote. Download a copy and read
it in:

```python
text = Path("sample_data/your_text.txt").read_text(encoding="utf-8")
candidates = lg.rank_candidates(text)
```

Then read the list with a pencil, the way you would have anyway — but starting from
twelve words instead of six hundred.